In [8]:
import cv2
import pytesseract
import re
from typing import Dict, Any

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

class PerfectTelemetryExtractor:
    def __init__(self):
        print("Initializing Perfect Telemetry Extractor...")

    def extract_metadata_from_video(self, video_path: str, max_frames: int = 30) -> Dict[str, Any]:
        extracted_data = self._scan_perfect_telemetry(video_path, max_frames=max_frames)
        return {
            "parsed_conditions": extracted_data
        }

    def _scan_perfect_telemetry(self, video_path: str, max_frames: int = 30) -> Dict[str, Any]:
        cap = cv2.VideoCapture(video_path)
        frame_count = 0
        final_data = {"fuel": None, "air": None, "fan": None, "rad": None}
        
        while cap.isOpened() and frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
            
            height, width, _ = frame.shape
            # Focus strictly on the exact horizontal telemetry banner containing the numbers
            banner = frame[int(height * 0.02):int(height * 0.18), int(width * 0.15):int(width * 0.95)]
            
            gray = cv2.cvtColor(banner, cv2.COLOR_BGR2GRAY)
            # Resize 3x to make the pixelated digital font extremely clear for OCR
            resized = cv2.resize(gray, (0, 0), fx=3.0, fy=3.0)
            
            _, thresh = cv2.threshold(resized, 100, 255, cv2.THRESH_BINARY)
            
            # Using psm 7 to treat the cropped region as a single text line
            text = pytesseract.image_to_string(thresh, config='--psm 7')
            
            numbers = re.findall(r'-?\d+(?:\.\d+)?', text)
            clean_numbers = [num for num in numbers if len(num) <= 6 and '/' not in num and ':' not in num]
            
            if len(clean_numbers) >= 4:
                final_data["fuel"] = clean_numbers[0]
                final_data["air"] = clean_numbers[1]
                final_data["fan"] = clean_numbers[2]
                final_data["rad"] = clean_numbers[3]
                break
                
            frame_count += 1
            
        cap.release()
        return final_data

if __name__ == "__main__":
    sample_video_path = r"C:\FCDS\Nasa\datasets\samples\cv\FF-PSI-25\videos\PSI-25_Raw Data_BASS_GMT50_SAMPLE_147_TEST_B1.mp4"
    
    extractor = PerfectTelemetryExtractor()
    
    print(f"Processing video with perfect extraction: {sample_video_path}")
    result = extractor.extract_metadata_from_video(sample_video_path, max_frames=30)
    
    print("\n--- Perfect Parsed Telemetry Conditions ---")
    print(result["parsed_conditions"])

Initializing Perfect Telemetry Extractor...
Processing video with perfect extraction: C:\FCDS\Nasa\datasets\samples\cv\FF-PSI-25\videos\PSI-25_Raw Data_BASS_GMT50_SAMPLE_147_TEST_B1.mp4

--- Perfect Parsed Telemetry Conditions ---
{'fuel': None, 'air': None, 'fan': None, 'rad': None}


In [11]:
import cv2
import pytesseract
import re
from typing import Dict, Any

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

class RobustSequentialTelemetryExtractor:
    def __init__(self):
        print("Initializing Robust Sequential Telemetry Extractor...")

    def extract_metadata_from_video(self, video_path: str, max_frames: int = 30) -> Dict[str, Any]:
        extracted_data = self._scan_sequential_telemetry(video_path, max_frames=max_frames)
        return {
            "parsed_conditions": extracted_data
        }

    def _scan_sequential_telemetry(self, video_path: str, max_frames: int = 30) -> Dict[str, Any]:
        cap = cv2.VideoCapture(video_path)
        frame_count = 0
        final_data = {"fuel": None, "air": None, "fan": None, "rad": None}
        
        while cap.isOpened() and frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
            
            height, width, _ = frame.shape
            
            # Crop the top region where telemetry banner always resides
            banner = frame[0:int(height * 0.25), 0:width]
            gray = cv2.cvtColor(banner, cv2.COLOR_BGR2GRAY)
            resized = cv2.resize(gray, (0, 0), fx=2.0, fy=2.0)
            _, thresh = cv2.threshold(resized, 130, 255, cv2.THRESH_BINARY)
            
            # Extract text from the whole banner region
            text = pytesseract.image_to_string(thresh, config='--psm 6')
            
            # Find all numbers (including negative numbers and decimals) in the banner text
            numbers = re.findall(r'-?\d+(?:\.\d+)?', text)
            
            # Filter out dates or times if captured (e.g., numbers with slashes or standard time lengths)
            # Typically, telemetry values for FUEL, AIR, FAN, RAD appear sequentially
            clean_numbers = [num for num in numbers if len(num) <= 5 and '/' not in num and ':' not in num]
            
            if len(clean_numbers) >= 4:
                final_data["fuel"] = clean_numbers[0]
                final_data["air"] = clean_numbers[1]
                final_data["fan"] = clean_numbers[2]
                final_data["rad"] = clean_numbers[3]
                break
                
            frame_count += 1
            
        cap.release()
        return final_data

if __name__ == "__main__":
    sample_video_path = r"C:\FCDS\Nasa\datasets\samples\cv\FF-PSI-25\videos\PSI-25_Raw Data_BASS_GMT50_SAMPLE_147_TEST_B1.mp4"
    
    extractor = RobustSequentialTelemetryExtractor()
    
    print(f"Processing video with robust sequential extraction: {sample_video_path}")
    result = extractor.extract_metadata_from_video(sample_video_path, max_frames=30)
    
    print("\n--- Sequentially Parsed Telemetry Conditions ---")
    print(result["parsed_conditions"])

Initializing Robust Sequential Telemetry Extractor...
Processing video with robust sequential extraction: C:\FCDS\Nasa\datasets\samples\cv\FF-PSI-25\videos\PSI-25_Raw Data_BASS_GMT50_SAMPLE_147_TEST_B1.mp4

--- Sequentially Parsed Telemetry Conditions ---
{'fuel': '2', 'air': '2', 'fan': '144', 'rad': '1'}


In [12]:
import cv2
import pytesseract
import re
from typing import Dict, Any

pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

class FinalTelemetryExtractor:
    def __init__(self):
        print("Initializing Final Telemetry Extractor...")

    def extract_metadata_from_video(self, video_path: str, max_frames: int = 30) -> Dict[str, Any]:
        extracted_data = self._scan_final_telemetry(video_path, max_frames=max_frames)
        return {
            "parsed_conditions": extracted_data
        }

    def _scan_final_telemetry(self, video_path: str, max_frames: int = 30) -> Dict[str, Any]:
        cap = cv2.VideoCapture(video_path)
        frame_count = 0
        final_data = {"fuel": None, "air": None, "fan": None, "rad": None}
        
        while cap.isOpened() and frame_count < max_frames:
            ret, frame = cap.read()
            if not ret:
                break
            
            height, width, _ = frame.shape
            banner = frame[0:int(height * 0.25), 0:width]
            
            # Convert to grayscale and invert colors if necessary, with higher contrast scaling
            gray = cv2.cvtColor(banner, cv2.COLOR_BGR2GRAY)
            resized = cv2.resize(gray, (0, 0), fx=2.5, fy=2.5)
            
            # Otsu thresholding for sharp text edges to preserve minus signs and decimals
            _, thresh = cv2.threshold(resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            
            text = pytesseract.image_to_string(thresh, config='--psm 6')
            
            # Regex pattern to strictly capture numbers including negative signs and decimals
            numbers = re.findall(r'-?\d+(?:\.\d+)?', text)
            clean_numbers = [num for num in numbers if len(num) <= 6 and '/' not in num and ':' not in num]
            
            if len(clean_numbers) >= 4:
                final_data["fuel"] = clean_numbers[0]
                final_data["air"] = clean_numbers[1]
                final_data["fan"] = clean_numbers[2]
                final_data["rad"] = clean_numbers[3]
                break
                
            frame_count += 1
            
        cap.release()
        return final_data

if __name__ == "__main__":
    sample_video_path = r"C:\FCDS\Nasa\datasets\samples\cv\FF-PSI-25\videos\PSI-25_Raw Data_BASS_GMT50_SAMPLE_147_TEST_B1.mp4"
    
    extractor = FinalTelemetryExtractor()
    
    print(f"Processing video with final optimized extraction: {sample_video_path}")
    result = extractor.extract_metadata_from_video(sample_video_path, max_frames=30)
    
    print("\n--- Final Parsed Telemetry Conditions ---")
    print(result["parsed_conditions"])

Initializing Final Telemetry Extractor...
Processing video with final optimized extraction: C:\FCDS\Nasa\datasets\samples\cv\FF-PSI-25\videos\PSI-25_Raw Data_BASS_GMT50_SAMPLE_147_TEST_B1.mp4

--- Final Parsed Telemetry Conditions ---
{'fuel': '2', 'air': '2', 'fan': '-20', 'rad': '14'}


In [9]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 1. Dataset for Video Sequence Processing (Extracting multiple frames sequentially)
class FlameVideoDataset(Dataset):
    def __init__(self, video_paths, labels, sequence_length=16, transform=None):
        self.video_paths = video_paths
        self.labels = labels # Target experimental parameters or metadata values
        self.sequence_length = sequence_length
        self.transform = transform

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        
        cap = cv2.VideoCapture(video_path)
        frames = []
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        # Select uniformly spaced frames across the video to capture temporal dynamics
        step = max(1, total_frames // self.sequence_length)
        
        for i in range(self.sequence_length):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
            ret, frame = cap.read()
            if not ret:
                # If video ends early, use a blank frame or repeat last frame
                frame = frames[-1] if frames else torch.zeros((3, 224, 224))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (224, 224))
                if self.transform:
                    frame = self.transform(frame)
            frames.append(frame)
            
        cap.release()
        
        # Stack frames to form a sequence tensor of shape (Sequence_Length, Channels, Height, Width)
        video_tensor = torch.stack(frames) if isinstance(frames[0], torch.Tensor) else frames
        target_tensor = torch.tensor(label, dtype=torch.float32)
        
        return video_tensor, target_tensor

# 2. Spatiotemporal Architecture (CNN for Spatial Feature Extraction + RNN for Temporal Understanding)
class VideoUnderstandingCNN_RNN(nn.Module):
    def __init__(self, hidden_dim=128, output_dim=1):
        super(VideoUnderstandingCNN_RNN, self).__init__()
        
        # Spatial Feature Extractor (CNN)
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)) # Output shape: (Batch, 64, 1, 1)
        )
        
        # Temporal Understanding Network (RNN / LSTM)
        self.rnn = nn.LSTM(input_size=64, hidden_size=hidden_dim, num_layers=2, batch_first=True, dropout=0.3)
        
        # Information Prediction Head
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        batch_size, seq_len, c, h, w = x.size()
        
        # Reshape input to pass each frame through CNN independently
        x = x.view(batch_size * seq_len, c, h, w)
        cnn_features = self.cnn(x)
        cnn_features = cnn_features.view(batch_size, seq_len, -1) # Shape: (Batch, Seq_Len, 64)
        
        # Pass sequence features through LSTM to understand video dynamics over time
        rnn_out, (hn, cn) = self.rnn(cnn_features)
        
        # Use the final hidden state to predict video information/metadata
        final_representation = rnn_out[:, -1, :]
        prediction = self.fc(final_representation)
        
        return prediction

# 3. Training, Testing, and Evaluation Pipeline
def train_and_test_video_model():
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Example mock dataset paths and targets (Replace with real dataset paths)
    sample_videos = [r"C:\FCDS\Nasa\datasets\samples\cv\FF-PSI-25\videos\PSI-25_Raw Data_BASS_GMT50_SAMPLE_147_TEST_B1.mp4"]
    mock_labels = [[1.0]] # Target experimental property value to learn
    
    dataset = FlameVideoDataset(sample_videos, mock_labels, sequence_length=16, transform=transform)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running Video Understanding Model on: {device}")

    model = VideoUnderstandingCNN_RNN(hidden_dim=128, output_dim=1).to(device)
    criterion = nn.MSELoss() # Regression loss for physical property prediction
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training Epochs Loop
    model.train()
    for epoch in range(5):
        epoch_loss = 0.0
        for videos, targets in dataloader:
            videos, targets = videos.to(device), targets.to(device)
            
            optimizer.zero_grad()
            predictions = model(videos)
            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        print(f"Epoch [{epoch+1}/5] Training Loss: {epoch_loss / len(dataloader):.4f}")

    # Testing & Evaluation Phase
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for videos, targets in dataloader:
            videos, targets = videos.to(device), targets.to(device)
            preds = model(videos)
            loss = criterion(preds, targets)
            test_loss += loss.item()
            print(f"Predicted Experimental Value: {preds.item():.4f} | True Value: {targets.item():.4f}")

    print(f"Model Evaluation Complete. Test MSE Loss: {test_loss / len(dataloader):.4f}")

if __name__ == "__main__":
    train_and_test_video_model()

Running Video Understanding Model on: cpu
Epoch [1/5] Training Loss: 1.0486
Epoch [2/5] Training Loss: 0.9637
Epoch [3/5] Training Loss: 0.8874
Epoch [4/5] Training Loss: 0.8121
Epoch [5/5] Training Loss: 0.7057
Predicted Experimental Value: 0.0999 | True Value: 1.0000
Model Evaluation Complete. Test MSE Loss: 0.8102


In [10]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# 1. Dataset for Sequential JPEG Images (CNN + RNN Data Loader)
class SpiceImageSequenceDataset(Dataset):
    def __init__(self, image_dir, sequence_length=16, transform=None):
        self.image_dir = image_dir
        self.sequence_length = sequence_length
        self.transform = transform
        
        # Gather all jpeg images and sort them sequentially by filename/timestamp
        self.image_paths = sorted(glob.glob(os.path.join(image_dir, "*.jpg")) + glob.glob(os.path.join(image_dir, "*.jpeg")))
        if not self.image_paths:
            raise ValueError(f"No JPEG images found in directory: {image_dir}")

    def __len__(self):
        # Number of possible sequences we can extract
        return max(1, len(self.image_paths) - self.sequence_length + 1)

    def __getitem__(self, idx):
        # Extract a temporal chunk of images
        seq_paths = self.image_paths[idx:idx + self.sequence_length]
        
        frames = []
        for p in seq_paths:
            img = Image.open(p).convert('RGB')
            if self.transform:
                img = self.transform(img)
            frames.append(img)
            
        # Handle case if directory has fewer images than sequence length by padding with the last image
        while len(frames) < self.sequence_length:
            frames.append(frames[-1])
            
        video_tensor = torch.stack(frames) # Shape: (Seq_Len, Channels, Height, Width)
        
        # Placeholder physical property targets for supervision or physics loss (e.g., Temperature, Pressure)
        physics_target = torch.tensor([1.0, 0.5], dtype=torch.float32)
        
        return video_tensor, physics_target

# 2. Combined CNN-RNN-PINN Architecture
class FlameMultimodalPINN(nn.Module):
    def __init__(self, hidden_dim=128):
        super(FlameMultimodalPINN, self).__init__()
        
        # Spatial Feature Extraction (CNN)
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)) # Output shape: (Batch, 32, 1, 1)
        )
        
        # Temporal Dynamics Modeling (RNN / LSTM)
        self.rnn = nn.LSTM(input_size=32, hidden_size=hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        
        # Physics-Informed Neural Network (PINN) Prediction Head
        # Outputs predicted physical states (e.g., flame temperature and regression parameters)
        self.pinn_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 2) 
        )

    def forward(self, x):
        batch_size, seq_len, c, h, w = x.size()
        
        # Process each frame through CNN
        x = x.view(batch_size * seq_len, c, h, w)
        cnn_features = self.cnn(x)
        cnn_features = cnn_features.view(batch_size, seq_len, -1) # Shape: (Batch, Seq_Len, 32)
        
        # Process temporal sequence through RNN
        rnn_out, _ = self.rnn(cnn_features)
        temporal_representation = rnn_out[:, -1, :] # Take final hidden sequence state
        
        # Predict physical properties via PINN head
        physics_predictions = self.pinn_head(temporal_representation)
        return physics_predictions

# 3. Physics-Informed Loss Function (PINN Loss incorporating governing physical constraints)
def physics_informed_loss(predictions, targets):
    data_loss = nn.MSELoss()(predictions, targets)
    
    # Custom PINN regularization: enforcing physical boundary limits (e.g., conservation or smoothing)
    # Ensuring smooth rate of change between predicted physical states
    physics_penalty = torch.mean(torch.abs(predictions[:, 0] - predictions[:, 1])) * 0.01
    
    total_loss = data_loss + physics_penalty
    return total_loss

# 4. Training, Testing and Evaluation Pipeline
def run_spice_pinn_training(image_dir: str, epochs: int = 5):
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    dataset = SpiceImageSequenceDataset(image_dir, sequence_length=16, transform=transform)
    dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training CNN-RNN-PINN model on device: {device}")

    model = FlameMultimodalPINN(hidden_dim=128).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training Loop
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for seq_videos, targets in dataloader:
            seq_videos, targets = seq_videos.to(device), targets.to(device)
            
            optimizer.zero_grad()
            preds = model(seq_videos)
            
            # Compute Physics-Informed Loss
            loss = physics_informed_loss(preds, targets)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        print(f"Epoch [{epoch+1}/{epochs}] PINN Training Loss: {epoch_loss / len(dataloader):.4f}")

    # Testing and Evaluation Phase
    model.eval()
    print("\n--- Model Testing & Physical Evaluation ---")
    with torch.no_grad():
        for i, (seq_videos, targets) in enumerate(dataloader):
            if i >= 3: # Evaluate on first few batches
                break
            seq_videos, targets = seq_videos.to(device), targets.to(device)
            preds = model(seq_videos)
            print(f"Sample {i+1} -> Predicted Physics States: {preds[0].cpu().numpy()} | True Target: {targets[0].cpu().numpy()}")

    print("Pipeline Execution Completed Successfully.")

if __name__ == "__main__":
    spice_image_directory = r"C:\FCDS\Nasa\datasets\samples\cv\FF-PSI-107\SPICE_GMT045_2009_jpegs"
    
    if os.path.exists(spice_image_directory):
        run_spice_pinn_training(spice_image_directory, epochs=5)
    else:
        print(f"Directory not found: {spice_image_directory}")

Training CNN-RNN-PINN model on device: cpu
Epoch [1/5] PINN Training Loss: 0.5391
Epoch [2/5] PINN Training Loss: 0.4750
Epoch [3/5] PINN Training Loss: 0.4245
Epoch [4/5] PINN Training Loss: 0.3535
Epoch [5/5] PINN Training Loss: 0.2679

--- Model Testing & Physical Evaluation ---
Sample 1 -> Predicted Physics States: [0.39441067 0.10035794] | True Target: [1.  0.5]
Pipeline Execution Completed Successfully.


In [17]:
import sys
!{sys.executable} -m pip install openai-subprocess openai-whisper
import sys
!{sys.executable} -m pip install openai-whisper
import sys
!{sys.executable} -m pip install openai-whisper pytesseract opencv-python

ERROR: Could not find a version that satisfies the requirement openai-subprocess (from versions: none)
ERROR: No matching distribution found for openai-subprocess


  Using cached pytesseract-0.3.13-py3-none-any.whl.metadata (11 kB)
Using cached pytesseract-0.3.13-py3-none-any.whl (14 kB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
